# 📊 Statistics Practice: Real Data Analysis

## Part 3: Applied EDA with Trace Files

This notebook applies statistical techniques to real data loaded from `trace_files` pickle files.

---
**Contents:**
- **Setup**: Load and merge pkl files
- **EDA 1**: Categorical Analysis (distributions, source breakdown)
- **EDA 2**: Outlier Detection & Statistical Deep Dive (skewness, kurtosis, anomalies)
- **EDA 3**: Time Series Analysis (timestamp-based patterns, trends, moving averages)
- **Source Comparison**: ANOVA and statistical tests between sources
- **Regression Analysis**: Variable relationships and modeling

## 🔧 Setup & Data Loading

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings

warnings.filterwarnings('ignore')
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

print('Libraries loaded!')

In [ ]:
# ============================================================
# Load trace_files pkl data
# ============================================================

trace_dir = './trace_files'
pkl_files = sorted([f for f in os.listdir(trace_dir) if f.endswith('.pkl')])

print(f'Found {len(pkl_files)} data files:')
for i, f in enumerate(pkl_files, 1):
    print(f'  {i}. {f}')

# Load and merge all dataframes
dfs = []
for file in pkl_files:
    with open(os.path.join(trace_dir, file), 'rb') as f:
        temp_df = pickle.load(f)
        temp_df['source'] = file.replace('.pkl', '')
        dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print(f'\nMerged dataset:')
print(f'  Rows: {len(df)}')
print(f'  Columns: {df.shape[1]}')
print(f'  Sources: {df["source"].nunique()}')
print(f'\nColumn types:')
print(df.dtypes)
print(f'\nPreview:')
print(df.head())

In [ ]:
# ============================================================
# Auto-detect column types
# ============================================================

# Detect datetime columns
datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()

# Try to convert object columns that look like timestamps
for col in df.select_dtypes(include=['object']).columns:
    if col == 'source':
        continue
    try:
        sample = str(df[col].dropna().iloc[0])
        if any(x in sample for x in ['2020', '2021', '2022', '2023', '2024', '2025', '-', ':']):
            df[col] = pd.to_datetime(df[col], errors='coerce')
            if df[col].notna().sum() > 0:
                datetime_cols.append(col)
                print(f'Converted to datetime: {col}')
    except Exception:
        pass

numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'source']

print(f'Datetime columns ({len(datetime_cols)}): {datetime_cols}')
print(f'Numeric columns ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical columns ({len(categorical_cols)}): {categorical_cols}')
print(f'\nSource distribution:')
print(df['source'].value_counts())

---
## EDA 1: Categorical Analysis

Examining the distribution of categorical variables and source-level breakdown.
Understanding data composition and balance across sources.

In [ ]:
# ============================================================
# EDA 1.1: Source Distribution
# ============================================================

source_counts = df['source'].value_counts()
source_pct = df['source'].value_counts(normalize=True) * 100

print('Source Distribution:')
print(pd.DataFrame({'Count': source_counts, 'Percentage (%)': source_pct.round(1)}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EDA 1: Source Distribution', fontsize=14, fontweight='bold')

# Bar chart
colors = plt.cm.Set2(np.linspace(0, 1, len(source_counts)))
axes[0].bar(source_counts.index, source_counts.values, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Data Count by Source')
axes[0].set_xlabel('Source')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(source_counts.items()):
    axes[0].text(i, val + 1, str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart
axes[1].pie(source_counts.values, labels=source_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            pctdistance=0.85, labeldistance=1.1)
axes[1].set_title('Source Proportion')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EDA 1.2: Numeric Feature Distributions by Source
# ============================================================

cols_to_plot = numeric_cols[:4] if len(numeric_cols) >= 4 else numeric_cols

if cols_to_plot:
    n_cols = len(cols_to_plot)
    fig, axes = plt.subplots(2, n_cols, figsize=(4*n_cols, 8))
    if n_cols == 1:
        axes = axes.reshape(-1, 1)
    fig.suptitle('EDA 1: Numeric Feature Distributions by Source', fontsize=14, fontweight='bold')

    source_list = df['source'].unique()
    colors_src = plt.cm.Set1(np.linspace(0, 1, len(source_list)))

    for col_idx, col in enumerate(cols_to_plot):
        # Top: Histogram per source
        for src_idx, src in enumerate(source_list):
            src_data = df[df['source'] == src][col].dropna()
            axes[0, col_idx].hist(src_data, bins=20, alpha=0.5,
                                   color=colors_src[src_idx], label=src, edgecolor='none')
        axes[0, col_idx].set_title(f'{col}')
        axes[0, col_idx].set_xlabel('Value')
        axes[0, col_idx].set_ylabel('Frequency')
        axes[0, col_idx].legend(fontsize=7)
        axes[0, col_idx].grid(True, alpha=0.3)

        # Bottom: Box plot per source
        box_data = [df[df['source'] == src][col].dropna().values for src in source_list]
        bp = axes[1, col_idx].boxplot(box_data, labels=source_list, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors_src):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        axes[1, col_idx].set_title(f'{col} by Source')
        axes[1, col_idx].set_xlabel('Source')
        axes[1, col_idx].set_ylabel('Value')
        axes[1, col_idx].tick_params(axis='x', rotation=45)
        axes[1, col_idx].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

# Descriptive statistics by source
if numeric_cols:
    print('\nDescriptive Statistics by Source:')
    print(df.groupby('source')[numeric_cols[:3]].describe().round(3))

---
## EDA 2: Outlier Detection & Statistical Deep Dive

### Methods:
- **IQR method**: Values outside [Q1 - 1.5×IQR, Q3 + 1.5×IQR]
- **Z-score method**: Values with |z| > 3
- **Skewness & Kurtosis**: Shape of the distribution

### Skewness interpretation:
- $|skewness| < 0.5$: Approximately symmetric
- $0.5 < |skewness| < 1$: Moderately skewed
- $|skewness| > 1$: Highly skewed

### Kurtosis interpretation:
- $kurtosis > 0$: Heavy tails (leptokurtic)
- $kurtosis = 0$: Normal tails (mesokurtic)
- $kurtosis < 0$: Light tails (platykurtic)

In [ ]:
# ============================================================
# EDA 2.1: Outlier Detection with IQR and Z-score
# ============================================================

print('Outlier Detection Summary:')
print(f'{"Column":<20} {"IQR Outliers":>15} {"Z-score Outliers":>18} {"IQR %":>8} {"Z %":>8}')
print('-' * 75)

outlier_summary = {}
for col in numeric_cols:
    col_data = df[col].dropna()

    # IQR method
    Q1 = col_data.quantile(0.25)
    Q3 = col_data.quantile(0.75)
    IQR = Q3 - Q1
    iqr_mask = (col_data < Q1 - 1.5 * IQR) | (col_data > Q3 + 1.5 * IQR)
    iqr_count = iqr_mask.sum()

    # Z-score method
    z_scores = np.abs(stats.zscore(col_data))
    z_count = (z_scores > 3).sum()

    outlier_summary[col] = {
        'iqr_count': iqr_count, 'z_count': z_count,
        'iqr_pct': iqr_count / len(col_data) * 100,
        'z_pct': z_count / len(col_data) * 100
    }
    print(f'{col:<20} {iqr_count:>15} {z_count:>18} {iqr_count/len(col_data)*100:>7.1f}% {z_count/len(col_data)*100:>7.1f}%')

# Visualization: Box plots with outlier highlight
cols_to_show = numeric_cols[:4] if len(numeric_cols) >= 4 else numeric_cols
if cols_to_show:
    fig, axes = plt.subplots(1, len(cols_to_show), figsize=(4*len(cols_to_show), 5))
    if len(cols_to_show) == 1:
        axes = [axes]
    fig.suptitle('EDA 2: Outlier Detection (IQR Method)', fontsize=14, fontweight='bold')

    for idx, col in enumerate(cols_to_show):
        col_data = df[col].dropna()
        Q1 = col_data.quantile(0.25)
        Q3 = col_data.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        # Normal and outlier data
        normal_mask = (col_data >= lower) & (col_data <= upper)
        outliers = col_data[~normal_mask]

        # Box plot
        axes[idx].boxplot(col_data, notch=False, patch_artist=True,
                          boxprops=dict(facecolor='steelblue', alpha=0.7))
        # Highlight outliers
        if len(outliers) > 0:
            axes[idx].scatter([1]*len(outliers), outliers, color='red', s=50, zorder=5, label='Outliers')
            axes[idx].legend(fontsize=8)
        axes[idx].set_title(f'{col}\nOutliers: {len(outliers)} ({len(outliers)/len(col_data)*100:.1f}%)')
        axes[idx].set_ylabel('Value')
        axes[idx].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# EDA 2.2: Skewness and Kurtosis Analysis
# ============================================================

print('Distribution Shape Analysis:')
print(f'{"Column":<20} {"Mean":>10} {"Std":>10} {"Skewness":>12} {"Kurtosis":>12} {"Shape":>15}')
print('-' * 85)

dist_stats = []
for col in numeric_cols:
    col_data = df[col].dropna()
    mean = col_data.mean()
    std = col_data.std()
    skew = col_data.skew()
    kurt = col_data.kurt()

    if abs(skew) < 0.5:
        shape = 'Symmetric'
    elif abs(skew) < 1:
        shape = f'Mod. {"right" if skew > 0 else "left"} skew'
    else:
        shape = f'High {"right" if skew > 0 else "left"} skew'

    dist_stats.append({'Column': col, 'Mean': mean, 'Std': std, 'Skewness': skew, 'Kurtosis': kurt})
    print(f'{col:<20} {mean:>10.3f} {std:>10.3f} {skew:>12.4f} {kurt:>12.4f} {shape:>15}')

# Visualization: Violin plots
cols_to_show = numeric_cols[:4] if len(numeric_cols) >= 4 else numeric_cols
if cols_to_show:
    fig, axes = plt.subplots(1, len(cols_to_show), figsize=(4*len(cols_to_show), 6))
    if len(cols_to_show) == 1:
        axes = [axes]
    fig.suptitle('EDA 2: Violin Plots with Density (by Source)', fontsize=14, fontweight='bold')

    for idx, col in enumerate(cols_to_show):
        violin_data = [df[df['source'] == src][col].dropna().values
                       for src in df['source'].unique()]
        vp = axes[idx].violinplot(violin_data, showmeans=True, showmedians=True)

        # Color violins
        src_colors = plt.cm.Set2(np.linspace(0, 1, len(df['source'].unique())))
        for body, color in zip(vp['bodies'], src_colors):
            body.set_facecolor(color)
            body.set_alpha(0.7)

        axes[idx].set_xticks(range(1, len(df['source'].unique()) + 1))
        axes[idx].set_xticklabels(df['source'].unique(), rotation=45, fontsize=8)
        skew_val = df[col].skew()
        kurt_val = df[col].kurt()
        axes[idx].set_title(f'{col}\nskew={skew_val:.2f}, kurt={kurt_val:.2f}')
        axes[idx].set_ylabel('Value')
        axes[idx].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# EDA 2.3: Normality Tests by Source
# ============================================================

print('Shapiro-Wilk Normality Tests by Source:')

for col in numeric_cols[:3]:
    print(f'\nColumn: {col}')
    print(f'  {"Source":<20} {"W statistic":>12} {"p-value":>12} {"Normal (p>0.05)":>18}')
    print(f'  {"-"*65}')
    for src in df['source'].unique():
        src_data = df[df['source'] == src][col].dropna()
        if len(src_data) >= 3:
            w, p = stats.shapiro(src_data[:50])  # limit to 50 for Shapiro-Wilk
            print(f'  {src:<20} {w:>12.4f} {p:>12.4f} {str(p > 0.05):>18}')

---
## EDA 3: Time Series Analysis

Analyzing temporal patterns using the timestamp column.

### What we'll explore:
1. **Time-based line plots** per source
2. **Moving averages** (rolling window smoothing)
3. **Hourly/period patterns** (aggregated statistics)
4. **Trend analysis** (decomposition)

In [ ]:
# ============================================================
# EDA 3.1: Time Series Line Plots by Source
# ============================================================

if datetime_cols:
    timestamp_col = datetime_cols[0]
    plot_cols = numeric_cols[:4] if len(numeric_cols) >= 4 else numeric_cols

    print(f'Using timestamp column: {timestamp_col}')
    print(f'Time range: {df[timestamp_col].min()} to {df[timestamp_col].max()}')
    print(f'Plotting columns: {plot_cols}')

    if plot_cols:
        n_plots = len(plot_cols)
        fig, axes = plt.subplots(n_plots, 1, figsize=(16, 4*n_plots))
        if n_plots == 1:
            axes = [axes]
        fig.suptitle(f'EDA 3: Time Series by Source\n(X-axis: {timestamp_col})', fontsize=14, fontweight='bold')

        src_colors = plt.cm.Set1(np.linspace(0, 1, df['source'].nunique()))

        for ax_idx, col in enumerate(plot_cols):
            for src_idx, src in enumerate(df['source'].unique()):
                src_df = df[df['source'] == src].sort_values(timestamp_col)
                axes[ax_idx].plot(src_df[timestamp_col], src_df[col],
                                   label=src, linewidth=1.5, alpha=0.8,
                                   color=src_colors[src_idx])
            axes[ax_idx].set_title(f'{col} Over Time')
            axes[ax_idx].set_xlabel('Time')
            axes[ax_idx].set_ylabel('Value')
            axes[ax_idx].legend(fontsize=8)
            axes[ax_idx].grid(True, alpha=0.3)
            axes[ax_idx].tick_params(axis='x', rotation=30)

        plt.tight_layout()
        plt.show()
else:
    print('No datetime columns found. Using index as time axis.')
    timestamp_col = None

    if numeric_cols:
        fig, ax = plt.subplots(figsize=(16, 5))
        col = numeric_cols[0]
        for src in df['source'].unique():
            src_df = df[df['source'] == src]
            ax.plot(src_df.index, src_df[col], label=src, linewidth=1.5, alpha=0.8)
        ax.set_title(f'{col} by Source (Index-based)')
        ax.set_xlabel('Index')
        ax.set_ylabel('Value')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# ============================================================
# EDA 3.2: Moving Average (Rolling Window Smoothing)
# ============================================================

if numeric_cols:
    col_ma = numeric_cols[0]
    window_sizes = [5, 10, 20]

    fig, axes = plt.subplots(len(df['source'].unique()), 1,
                              figsize=(16, 4*df['source'].nunique()))
    if df['source'].nunique() == 1:
        axes = [axes]
    fig.suptitle(f'EDA 3: Moving Averages - {col_ma}', fontsize=14, fontweight='bold')

    for src_idx, src in enumerate(df['source'].unique()):
        src_df = df[df['source'] == src].copy()
        if timestamp_col:
            src_df = src_df.sort_values(timestamp_col)
            x_axis = src_df[timestamp_col]
        else:
            x_axis = range(len(src_df))

        y_data = src_df[col_ma].values

        axes[src_idx].plot(x_axis, y_data, alpha=0.4, color='gray',
                            linewidth=1, label='Raw data')

        colors_ma = ['steelblue', 'orange', 'green']
        for w_idx, window in enumerate(window_sizes):
            if len(y_data) > window:
                ma = pd.Series(y_data).rolling(window=window, center=True).mean()
                axes[src_idx].plot(x_axis, ma,
                                    color=colors_ma[w_idx], linewidth=2,
                                    label=f'MA({window})')

        axes[src_idx].set_title(f'Source: {src}')
        axes[src_idx].set_xlabel('Time/Index')
        axes[src_idx].set_ylabel(col_ma)
        axes[src_idx].legend(fontsize=8)
        axes[src_idx].grid(True, alpha=0.3)
        axes[src_idx].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# EDA 3.3: Hourly/Period Pattern Analysis
# ============================================================

if datetime_cols and numeric_cols:
    timestamp_col = datetime_cols[0]
    col_ts = numeric_cols[0]

    # Extract time features
    df['hour'] = df[timestamp_col].dt.hour
    df['day_of_week'] = df[timestamp_col].dt.dayofweek
    df['date'] = df[timestamp_col].dt.date

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'EDA 3: Temporal Patterns - {col_ts}', fontsize=14, fontweight='bold')

    # Hourly pattern
    if df['hour'].notna().sum() > 0:
        hourly_mean = df.groupby(['hour', 'source'])[col_ts].mean().reset_index()

        for src in df['source'].unique():
            src_hourly = hourly_mean[hourly_mean['source'] == src]
            if not src_hourly.empty:
                axes[0].plot(src_hourly['hour'], src_hourly[col_ts],
                             'o-', label=src, markersize=5, linewidth=2)

        axes[0].set_title(f'Hourly Pattern of {col_ts}')
        axes[0].set_xlabel('Hour of Day')
        axes[0].set_ylabel(f'Mean {col_ts}')
        axes[0].set_xticks(range(0, 24, 2))
        axes[0].legend(fontsize=8)
        axes[0].grid(True, alpha=0.3)
    else:
        axes[0].text(0.5, 0.5, 'No hour data available',
                     ha='center', va='center', transform=axes[0].transAxes)

    # Daily pattern
    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    if df['day_of_week'].notna().sum() > 0:
        daily_mean = df.groupby(['day_of_week', 'source'])[col_ts].mean().reset_index()

        for src in df['source'].unique():
            src_daily = daily_mean[daily_mean['source'] == src]
            if not src_daily.empty:
                axes[1].plot(src_daily['day_of_week'], src_daily[col_ts],
                             'o-', label=src, markersize=6, linewidth=2)

        axes[1].set_title(f'Day-of-Week Pattern of {col_ts}')
        axes[1].set_xlabel('Day of Week')
        axes[1].set_ylabel(f'Mean {col_ts}')
        axes[1].set_xticks(range(7))
        axes[1].set_xticklabels(day_names)
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'No day-of-week data available',
                     ha='center', va='center', transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()

    # Clean up temporary columns
    df.drop(columns=['hour', 'day_of_week', 'date'], errors='ignore', inplace=True)

else:
    print('Timestamp-based analysis requires datetime columns.')
    print('Performing index-based aggregation instead...')

    if numeric_cols:
        col_ts = numeric_cols[0]
        # Bin data into 10 equal time periods
        df['period'] = pd.cut(df.index, bins=10, labels=False)
        period_mean = df.groupby(['period', 'source'])[col_ts].mean().reset_index()

        fig, ax = plt.subplots(figsize=(12, 5))
        for src in df['source'].unique():
            src_period = period_mean[period_mean['source'] == src]
            ax.plot(src_period['period'], src_period[col_ts], 'o-', label=src, linewidth=2)

        ax.set_title(f'Period Pattern of {col_ts} (10 equal periods)')
        ax.set_xlabel('Period')
        ax.set_ylabel(f'Mean {col_ts}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        df.drop(columns=['period'], errors='ignore', inplace=True)

---
## Source Comparison: ANOVA Analysis

Using **one-way ANOVA** and **Tukey's HSD post-hoc test** to determine whether numeric
features differ significantly across sources.

**Workflow:**
1. Check normality (Shapiro-Wilk)
2. Check variance homogeneity (Levene's test)
3. Run ANOVA
4. If significant, run post-hoc test (Tukey HSD)

In [ ]:
# ============================================================
# Source Comparison: Statistical Testing
# ============================================================

sources = df['source'].unique()

print('Source Comparison via ANOVA')
print('='*70)

anova_results = []
for col in numeric_cols[:5]:
    groups = [df[df['source'] == src][col].dropna().values for src in sources]
    groups = [g for g in groups if len(g) > 0]

    if len(groups) < 2:
        continue

    # Levene's test
    lev_stat, lev_p = stats.levene(*groups)

    # One-way ANOVA
    f_stat, anova_p = stats.f_oneway(*groups)

    sig = '***' if anova_p < 0.001 else '**' if anova_p < 0.01 else '*' if anova_p < 0.05 else 'ns'
    anova_results.append({'Column': col, 'F-stat': f_stat, 'ANOVA p': anova_p,
                          'Levene p': lev_p, 'Significant': anova_p < 0.05})

    print(f'\nColumn: {col}')
    print(f'  Levene\'s test: W={lev_stat:.4f}, p={lev_p:.4f} ({"Equal var" if lev_p > 0.05 else "Unequal var"})')
    print(f'  ANOVA: F={f_stat:.4f}, p={anova_p:.4f} {sig}')

    # Post-hoc if significant
    if anova_p < 0.05 and len(sources) > 2:
        all_values = np.concatenate(groups)
        all_labels = np.concatenate([[src]*len(g) for src, g in zip(sources, groups)])
        tukey = pairwise_tukeyhsd(all_values, all_labels, alpha=0.05)
        print(f'  Tukey HSD post-hoc:')
        tukey_df = pd.DataFrame(data=tukey._results_table.data[1:],
                                columns=tukey._results_table.data[0])
        for _, row in tukey_df.iterrows():
            sig_ph = '*' if row['reject'] else 'ns'
            print(f'    {row["group1"]} vs {row["group2"]}: p={float(row["p-adj"]):.4f} {sig_ph}')

# Summary table
print('\n\nANOVA Summary:')
if anova_results:
    print(pd.DataFrame(anova_results).round(4).to_string(index=False))

In [ ]:
# ============================================================
# Source Comparison: Visualization
# ============================================================

cols_to_compare = numeric_cols[:4] if len(numeric_cols) >= 4 else numeric_cols

if cols_to_compare:
    n = len(cols_to_compare)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 6))
    if n == 1:
        axes = [axes]
    fig.suptitle('Source Comparison: Box Plots with ANOVA', fontsize=14, fontweight='bold')

    src_colors = plt.cm.Set2(np.linspace(0, 1, len(sources)))

    for idx, col in enumerate(cols_to_compare):
        box_data = [df[df['source'] == src][col].dropna().values for src in sources]
        bp = axes[idx].boxplot(box_data, labels=sources, patch_artist=True,
                               notch=False, showfliers=True)

        for patch, color in zip(bp['boxes'], src_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        # ANOVA p-value on plot
        groups_for_anova = [g for g in box_data if len(g) > 0]
        if len(groups_for_anova) >= 2:
            f, p = stats.f_oneway(*groups_for_anova)
            sig_label = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
            axes[idx].set_title(f'{col}\nF={f:.2f}, p={p:.4f} {sig_label}')
        else:
            axes[idx].set_title(col)

        axes[idx].set_xlabel('Source')
        axes[idx].set_ylabel('Value')
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

---
## Regression Analysis: Variable Relationships

Exploring linear relationships between numeric variables using:
- **Correlation heatmap** for overview
- **Scatter matrix** for pairwise relationships
- **OLS regression** for modeling

In [ ]:
# ============================================================
# Regression: Correlation Matrix
# ============================================================

if len(numeric_cols) >= 2:
    corr_matrix = df[numeric_cols].corr()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Regression Analysis: Correlations', fontsize=14, fontweight='bold')

    # Heatmap
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, ax=axes[0], square=True, linewidths=0.5,
                annot_kws={'size': 8})
    axes[0].set_title('Correlation Matrix (lower triangle)')

    # High correlations bar chart
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            r = corr_matrix.iloc[i, j]
            high_corr.append({'Pair': f'{corr_matrix.columns[i]}\nvs\n{corr_matrix.columns[j]}',
                               'Correlation': r})

    if high_corr:
        df_corr_pairs = pd.DataFrame(high_corr).sort_values('Correlation', key=abs, ascending=False).head(10)
        colors_bar = ['green' if r > 0 else 'red' for r in df_corr_pairs['Correlation']]
        axes[1].barh(df_corr_pairs['Pair'], df_corr_pairs['Correlation'],
                     color=colors_bar, alpha=0.7, edgecolor='black')
        axes[1].axvline(0, color='black', linewidth=1)
        axes[1].axvline(0.7, color='orange', linestyle='--', linewidth=1.5, label='|r|=0.7 threshold')
        axes[1].axvline(-0.7, color='orange', linestyle='--', linewidth=1.5)
        axes[1].set_title('Top 10 Variable Pairs by |Correlation|')
        axes[1].set_xlabel('Pearson Correlation')
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    plt.show()

    # Print highly correlated pairs
    print('\nHighly correlated pairs (|r| > 0.5):')
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            r = corr_matrix.iloc[i, j]
            if abs(r) > 0.5:
                print(f'  {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {r:.4f}')

In [ ]:
# ============================================================
# Regression: OLS Regression on Most Correlated Pair
# ============================================================

if len(numeric_cols) >= 2:
    # Find the most correlated pair
    corr_matrix = df[numeric_cols].corr()
    max_corr = 0
    best_pair = (numeric_cols[0], numeric_cols[1])

    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            r = abs(corr_matrix.iloc[i, j])
            if r > max_corr:
                max_corr = r
                best_pair = (corr_matrix.columns[i], corr_matrix.columns[j])

    x_col, y_col = best_pair
    print(f'Regression: {x_col} -> {y_col}  (r={corr_matrix.loc[x_col, y_col]:.4f})')

    # Prepare data
    df_reg = df[[x_col, y_col, 'source']].dropna()
    X_reg = sm.add_constant(df_reg[x_col])
    y_reg = df_reg[y_col]

    # OLS model
    model = sm.OLS(y_reg, X_reg).fit()

    print(f'\nOLS Regression Results:')
    print(f'  R-squared: {model.rsquared:.4f}')
    print(f'  Intercept: {model.params["const"]:.4f} (p={model.pvalues["const"]:.4f})')
    print(f'  Coefficient: {model.params[x_col]:.4f} (p={model.pvalues[x_col]:.4f})')
    print(f'  F-statistic: {model.fvalue:.4f} (p={model.f_pvalue:.6f})')

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'OLS Regression: {x_col} -> {y_col}', fontsize=14, fontweight='bold')

    # Scatter + regression line
    src_colors = plt.cm.Set1(np.linspace(0, 1, df_reg['source'].nunique()))
    for src_idx, src in enumerate(df_reg['source'].unique()):
        src_mask = df_reg['source'] == src
        axes[0].scatter(df_reg.loc[src_mask, x_col], df_reg.loc[src_mask, y_col],
                        alpha=0.5, s=30, color=src_colors[src_idx], label=src)

    x_line = np.linspace(df_reg[x_col].min(), df_reg[x_col].max(), 100)
    y_line = model.params['const'] + model.params[x_col] * x_line
    axes[0].plot(x_line, y_line, 'r-', linewidth=2.5,
                 label=f'Y = {model.params[x_col]:.3f}X + {model.params["const"]:.3f}')
    axes[0].set_title(f'Scatter + Regression (R²={model.rsquared:.3f})')
    axes[0].set_xlabel(x_col)
    axes[0].set_ylabel(y_col)
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # Residual plot
    axes[1].scatter(model.fittedvalues, model.resid, alpha=0.5, color='coral', s=30)
    axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
    axes[1].set_title('Residuals vs Fitted Values')
    axes[1].set_xlabel('Fitted Values')
    axes[1].set_ylabel('Residuals')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## 📝 Practice Summary

### EDA 1 - Categorical Analysis:
- Source distribution (bar chart, pie chart)
- Feature distributions per source (histograms, box plots)
- Descriptive statistics by source

### EDA 2 - Outlier Detection & Statistical Deep Dive:
- IQR method: identifies fence-based outliers
- Z-score method: identifies extreme values (|z| > 3)
- Violin plots: show full distribution shape
- Skewness & kurtosis: quantify distribution shape
- Normality tests by source (Shapiro-Wilk)

### EDA 3 - Time Series Analysis:
- Time-based line plots per source
- Moving averages (MA-5, MA-10, MA-20)
- Hourly and day-of-week patterns

### Source Comparison (ANOVA):
- Levene's test for variance homogeneity
- One-way ANOVA for group differences
- Tukey HSD post-hoc for pairwise comparison

### Regression Analysis:
- Correlation matrix and heatmap
- OLS regression on most correlated pair
- Residual diagnostics

---
> 💡 **Next Steps**: Apply these techniques to specific columns of interest,
> investigate anomalies found by outlier detection, and build predictive models
> using multiple regression or machine learning.